# Suite SVD — The SystemVerilog Preprocessor dialect

`Expressions.SystemVerilog.Preprocessor` models Verilog's and SystemVerilog's compiler directives and macro usages
(IEEE 1800 clause 22), and the comments kept among them, as mbse-programs has them: a macro usage where an expression
stands, and the directives and comments where a constraint block's items, or `randomize() with`'s, are listed. Its
kinds are validated, bind nothing but a conditional directive's macros' names within its condition, cover every edition
of IEEE 1364 and IEEE 1800, print as mbse-programs prints them, and are refused by evaluation by name.

In [ ]:
from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.SystemVerilog import Assertions as SVAssertions, Constraints as SVConstraints, \
    Expressions as SystemVerilog, Patterns as SVPatterns, Preprocessor as SVPreprocessor, Types as SVTypes
from mbse.Expressions.SystemVerilog.Constraints import Expressions as SVC
from mbse.Expressions.SystemVerilog.Expressions import Evaluators as SVV, Expressions as SV, Text as SVText
from mbse.Expressions.SystemVerilog.Preprocessor import Expressions as PP, Text as PPText
from mbse.Expressions.Framework import Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators
from support import raises

store = Builders.Store()  # the SystemVerilog family's six dialects, the Preprocessor's install before the others'
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, SVPreprocessor.install, SystemVerilog.install,
                SVTypes.install, SVPatterns.install, SVAssertions.install, SVConstraints.install]:
    store.transaction(install)
for define in [SystemVerilog.define, SVPreprocessor.define]:  # SystemVerilog's types, and the Preprocessor's editions
    store.transaction(define)


def term(spec, dialect=PP.DIALECT):
    """The term value `spec` writes, the root of `dialect`'s holder written in a transaction."""
    with store.transaction():
        path = dialect.path.split(".")
        holder = getattr(getattr(store.Expressions, path[0]), path[1]).Expression
        return F.term(holder(lambda e: e.root(dialect.spec(spec))))


def validate(spec, bound=(), dialect=PP.DIALECT):
    return dialect.validate(term(spec, dialect), bound)


text = lambda spec, dialect=PP.DIALECT: SVText.ToText(term(spec, dialect))  # noqa: E731
union = lambda name: [b.name for b in store.registered(name).branches]  # noqa: E731: a union's branches
spelled = lambda word: lambda i: i.spelling(word)  # noqa: E731: an identifier, of its spelling
name = lambda word: lambda u: u.nameexpression(lambda n: n.name(lambda i: i.identifier(spelled(word))))  # noqa: E731
parenthesized = lambda inner: lambda u: u.parenthesizedexpression(lambda p: p.expression(inner))  # noqa: E731
comment = lambda words, block=False: lambda u: u.comment(lambda c: c.text(words).block(block))  # noqa: E731
x, y = name("x"), name("y")
condition = parenthesized(SV.binary("&&", name("A"), SV.unary("!", name("B"))))  # (A && !B)
ifdef = lambda u: u.ifdefdirective(lambda d: d.condition(condition).items([SV.binary(">", x, 0), comment(" low")])  # noqa: E731
    .branches([lambda b: b.name(spelled("C")).items([SV.binary("<", x, 9)])]).has_else(True))  # an empty `else

<a id="SVD-01"></a>
## SVD-01 · Its union and holder: its kinds and invalid syntax, the directives and comments a constraint block's items too, a holder refining the framework's, linked by HasRoot, round-tripped through JSON

In [ ]:
# Its Term: its kinds, mbse-programs', then invalid syntax; Constraints' Term lists the directives and comment beside
# the macro usage it lists as Expressions' Term does, and Expressions' branches.
assert PP.TAGS == ("macrousage", "includedirective", "definedirective", "undefdirective", "timescaledirective",
                   "defaultnettypedirective", "ifdefdirective", "disabledtext", "otherdirective", "comment")
assert union(PP.DIALECT.union) == [*PP.TAGS, "error", "missing"] and list(PP.DIALECT.kinds()) == [*PP.TAGS, "error", "missing"]
assert PP.DIRECTIVES == PP.TAGS[1:-1] and PP.HELPERS == ("ElsifDirective",)
assert union(SVC.DIALECT.union)[:10] == list(PP.TAGS) and set(SV.TAGS) <= set(union(SVC.DIALECT.union))
assert "macrousage" in union(SV.DIALECT.union) and "comment" not in union(SV.DIALECT.union)  # an expression holds none
kinds = {b.name: b.schema for b in store.registered(PP.DIALECT.union).branches}
held = {b.name: b.schema for b in store.registered(SVC.DIALECT.union).branches}
assert all(kinds[tag] is held[tag] for tag in PP.TAGS)  # one schema, a branch of each union that holds it
assert PP.DIALECT.name() == "SystemVerilog Preprocessor" and PP.DIALECT.tag == "systemverilog_preprocessor"
assert "systemverilog_preprocessor" in union("Expressions.Framework.Term")
# Its holder: a reference object refining the framework's Expression, its root narrowed to its Term.
Holder = store.registered("Expressions.SystemVerilog.Preprocessor.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is store.registered(PP.DIALECT.union) and Holder.validate() == []
with store.transaction():
    holder = store.Expressions.SystemVerilog.Preprocessor.Expression(lambda e: e.root(PP.DIALECT.spec(ifdef)))
    linked = store.Schemas.OfExpression(lambda s: s.name("svd.ifdef").expression(
        lambda a: a.insert(lambda e: e.expression(holder))))
assert Validators.Validate(store)(Holder, holder) == [] and PP.DIALECT.validate(holder, {"x"}) == []
assert F.dialect_of(F.term(holder)) == PP.DIALECT.union and F.kind_of(F.term(holder))[0] == "ifdefdirective"
assert F.same(JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)).root, holder.root)
assert [F.same(entry.expression, holder) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
with raises(Exception, match="'binaryexpression' is not a member of a union value"):  # its root holds its own terms
    with store.transaction():
        store.Expressions.SystemVerilog.Preprocessor.Expression(lambda e: e.root(PP.DIALECT.spec(SV.binary("+", x, 1))))

<a id="SVD-02"></a>
## SVD-02 · Its kinds and helpers: mbse-programs' properties in its order, a directive's branches holding Constraints' terms, its condition Expressions'

In [ ]:
props = lambda schema: list(schema.properties)  # noqa: E731
registered = lambda helper: store.registered(f"Expressions.SystemVerilog.Preprocessor.{helper}")  # noqa: E731
assert props(kinds["macrousage"]) == ["name", "arguments"]
assert props(kinds["includedirective"]) == ["path", "system"]
assert props(kinds["definedirective"]) == ["name", "function_like", "parameters", "body"]
assert props(kinds["undefdirective"]) == ["name"] and props(kinds["timescaledirective"]) == ["unit", "precision"]
assert props(kinds["defaultnettypedirective"]) == ["net_type"]
assert props(kinds["ifdefdirective"]) == ["negated", "name", "condition", "items", "branches", "alternative", "has_else"]
assert props(registered("ElsifDirective")) == ["name", "condition", "items"]
assert props(kinds["disabledtext"]) == ["text"] and props(kinds["otherdirective"]) == ["text"]
assert props(kinds["comment"]) == ["block", "text", "trailing"]
IFDEF = kinds["ifdefdirective"]
assert IFDEF.properties["condition"].schema is store.registered(SV.DIALECT.union)
assert IFDEF.properties["items"].schema.element is store.registered(SVC.DIALECT.union)
assert IFDEF.properties["alternative"].schema.element is store.registered(SVC.DIALECT.union)
assert IFDEF.properties["name"].schema is store.registered("Expressions.SystemVerilog.Expressions.Identifier")
assert F._parts(IFDEF) == (("condition",), ("items", "alternative"), ("negated", "name", "branches", "has_else"))
assert all(not registered(h).ref and registered(h).name.lower().split(".")[-1] not in kinds for h in PP.HELPERS)
# Its terms' nodes: the condition a slot, its items and its `else`'s two containers of terms, read apart (`lists`),
# each branch a helper holding its own. Its form gives each container's size, so that make puts each term back.
node = F.node(*F.kind_of(term(ifdef)))
assert len(node.many) == 2 and node.lists == {"items": node.many, "alternative": ()} and node.attributes["has_else"] is True
assert [b.kind for b in node.helpers["branches"]] == ["elsifdirective"] and node.helpers["branches"][0].lists == {
    "items": node.helpers["branches"][0].many} and node.helpers["branches"][0].helpers["name"].attributes == {"spelling": "C"}
assert "has_else" not in F.node(*F.kind_of(term(lambda u: u.ifdefdirective(lambda d: d.name(spelled("A")))))).attributes
form = F.form(term(ifdef))
assert form.attributes["items"] == F.Part("items", {}, 2) and form.attributes["alternative"] == F.Part("alternative", {}, 0)
assert F.same(term(PP.DIALECT.make(form)), term(ifdef))
# Given no sizes, a container holds none, but the last, which holds the rest; sizes that do not add up are refused.
bare = F.Form(form.kind, {k: v for k, v in form.attributes.items() if k not in ("items", "alternative")}, form.arguments)
assert F.node(*F.kind_of(term(PP.DIALECT.make(bare)))).lists == {"items": (), "alternative": node.many}
with raises(ValueError, match="an ifdefdirective's items, alternative hold 5 arguments, got 2"):
    term(PP.DIALECT.make(F.Form(form.kind, {**form.attributes, "items": F.Part("items", {}, 5)}, form.arguments)))

<a id="SVD-03"></a>
## SVD-03 · Its checks: what each kind requires, a default net type, an identifier's spelling, and a conditional directive's name or condition, with mbse-programs' messages

In [ ]:
undef = lambda word: lambda u: u.undefdirective(lambda d: d.name(spelled(word)))  # noqa: E731
nettype = lambda kind: lambda u: u.defaultnettypedirective(lambda d: d.net_type(kind))  # noqa: E731
ifdef_of = lambda build: lambda u: u.ifdefdirective(build)  # noqa: E731
for spec, problems in [
    (lambda u: u.macrousage(lambda m: m.name(spelled("WIDTH")).arguments("a, b")), []),
    (lambda u: u.macrousage(lambda m: m.arguments("a")), ["a macrousage needs a name"]),
    (lambda u: u.includedirective(lambda d: d.path("defs.svh").system(True)), []),
    (lambda u: u.includedirective(lambda d: d.path("")), ["an includedirective needs a path"]),
    (lambda u: u.definedirective(lambda d: d.name(spelled("F")).function_like(True).parameters([spelled("a"), spelled("2b")])),
     ["'2b' is not an identifier"]),
    (lambda u: u.definedirective(lambda d: d.body("1")), ["a definedirective needs a name"]),
    (undef("WIDTH"), []), (undef("9"), ["'9' is not an identifier"]),
    (lambda u: u.undefdirective(lambda d: d), ["an undefdirective needs a name"]),
    (lambda u: u.timescaledirective(lambda d: d.unit("1ns")), ["a timescaledirective needs a precision"]),
    (nettype("none"), []), (nettype("uwire"), []), (nettype("wirex"), ["DefaultNettypeDirective.net_type cannot be 'wirex'"]),
    (lambda u: u.defaultnettypedirective(lambda d: d), ["a defaultnettypedirective needs a net_type"]),
    (lambda u: u.disabledtext(lambda d: d.text("")), ["a disabledtext needs a text"]),
    (lambda u: u.otherdirective(lambda d: d), ["an otherdirective needs a text"]),
    (lambda u: u.otherdirective(lambda d: d.text("`resetall")), []),
    (comment(""), []), (lambda u: u.comment(lambda c: c), []),  # `//`: an empty comment is one
    (lambda u: u.comment(lambda c: c.block(1)), ["a comment's block must be a bool, got int"]),
    # a conditional directive: a name or a condition, in parentheses, of macros' names, !, &&, ||, -> and <->
    (ifdef_of(lambda d: d.name(spelled("A")).negated(True)), []),
    (ifdef_of(lambda d: d.condition(parenthesized(SV.binary("->", name("A"), SV.binary("<->", name("B"), parenthesized(name("C"))))))), []),
    (ifdef_of(lambda d: d), ["an IfdefDirective has a name or a condition"]),
    (ifdef_of(lambda d: d.name(spelled("A")).condition(condition)), ["an IfdefDirective has a name or a condition"]),
    (ifdef_of(lambda d: d.condition(name("A"))), ["an IfdefDirective's condition is in parentheses"]),
    (ifdef_of(lambda d: d.condition(parenthesized(SV.binary("+", name("A"), name("B"))))),
     ["an IfdefDirective's condition has macros' names, !, &&, ||, -> and <-> only"]),
    (ifdef_of(lambda d: d.condition(parenthesized(SV.unary("~", name("A"))))),
     ["an IfdefDirective's condition has macros' names, !, &&, ||, -> and <-> only"]),
    (ifdef_of(lambda d: d.condition(parenthesized(lambda u: u.unaryexpression(lambda r: r.operator("!").operand(name("A"))
        .attributes([lambda a: a.specs([lambda s: s.name(spelled("k"))])]))))),
     ["an IfdefDirective's condition has macros' names, !, &&, ||, -> and <-> only"]),
    (ifdef_of(lambda d: d.condition(parenthesized(lambda u: u.binaryexpression(lambda r: r.left(name("A")).operator("&&")
        .attributes([lambda a: a.specs([lambda s: s.name(spelled("k"))])]).right(name("B")))))),
     ["an IfdefDirective's condition has macros' names, !, &&, ||, -> and <-> only"]),
    (ifdef_of(lambda d: d.condition(parenthesized(lambda u: u.nameexpression(lambda n: n.name(lambda s: s.scopedname(
        lambda q: q.scope(lambda i: i.identifier(spelled("p"))).name(lambda i: i.identifier(spelled("A"))))))))),
     ["an IfdefDirective's condition has macros' names, !, &&, ||, -> and <-> only"]),
    (ifdef_of(lambda d: d.name(spelled("A")).branches([lambda b: b.items([])])), ["an ElsifDirective has a name or a condition"]),
    (ifdef_of(lambda d: d.name(spelled("A")).branches([lambda b: b.condition(name("B"))])),
     ["an ElsifDirective's condition is in parentheses"]),
    (ifdef_of(lambda d: d.name(spelled("1A"))), ["'1A' is not an identifier"]),
]:
    assert validate(spec) == problems, (problems, validate(spec))

<a id="SVD-04"></a>
## SVD-04 · What it binds: a conditional directive's macros' names, within its condition alone; its branches' items are read in their scope

In [ ]:
# A condition's names name macros, defined or not, never values: each is bound within its condition, an elsif's too,
# and not in the branches' items, whose names must be bound as anywhere else.
assert validate(ifdef) == ["items 0: left: nameexpression 'x' is not bound", "branches 0 argument 0: left: nameexpression 'x' is not bound"]
assert validate(ifdef, {"x"}) == [] and SY.free(term(ifdef), PP.DIALECT) == {"x"}
uses_macro = lambda u: u.ifdefdirective(lambda d: d.name(spelled("A")).items([name("A")]))  # noqa: E731: `ifdef A A `endif
assert validate(uses_macro) == ["items 0: nameexpression 'A' is not bound"]  # a name in its items is a value's
elsif = lambda u: u.ifdefdirective(lambda d: d.name(spelled("A")).branches([  # noqa: E731
    lambda b: b.condition(parenthesized(SV.binary("||", name("B"), name("C")))).items([y])]))
assert validate(elsif) == ["branches 0 argument 0: nameexpression 'y' is not bound"] and SY.free(term(elsif), PP.DIALECT) == {"y"}
assert not any(F.is_binder(kinds[tag]) for tag in PP.TAGS)  # none refines Let

<a id="SVD-05"></a>
## SVD-05 · Its kinds among the family's: directives and comments among a randomize() with's constraints and a constraint block's items, validated and printed from Expressions', Constraints' and its own holders

In [ ]:
# `randomize() with` lays out its constraints on one line, as mbse-programs does: a line comment ends its line, a
# directive takes lines of its own, a space between the others but not at a line's end or start.
randomize = lambda *items: lambda u: u.randomizewithexpression(  # noqa: E731
    lambda r: r.call(SV.method(x, "randomize")).items(list(items)))
mixed = randomize(SV.binary(">", x, 0), comment(" c"), comment(" b ", True), ifdef, SV.binary("<", x, 5),
                  lambda u: u.macrousage(lambda m: m.name(spelled("MORE"))))
assert validate(mixed, {"x"}, SV.DIALECT) == []
assert text(mixed, SV.DIALECT) == ("x.randomize() with { x > 0; // c\n/* b */\n`ifdef (A && !B)\nx > 0;\n// low\n"
                                   "`elsif C\nx < 9;\n`else\n`endif\nx < 5; `MORE }")
assert text(randomize(comment(" only")), SV.DIALECT) == "x.randomize() with { // only\n}"
assert text(randomize(lambda u: u.otherdirective(lambda d: d.text("`resetall"))), SV.DIALECT) == "x.randomize() with {\n`resetall\n}"
assert text(randomize(SV.binary(">", x, 0), SV.binary("<", x, 5)), SV.DIALECT) == "x.randomize() with { x > 0; x < 5; }"
# The same terms, rooted in Constraints' holder and its own; a directive's branches hold Constraints' terms, invalid
# syntax among them, and a nested directive.
nested = lambda u: u.ifdefdirective(lambda d: d.negated(True).name(spelled("SIM")).items([  # noqa: E731
    lambda i: i.timescaledirective(lambda t: t.unit("1ns").precision("1ps")), ifdef,
    lambda i: i.error(lambda e: e.text("x >"))]))
for dialect in (SVC.DIALECT, PP.DIALECT):
    assert validate(nested, {"x"}, dialect) == ["argument 2: invalid syntax 'x >'"]
    assert text(nested, dialect) == ("`ifndef SIM\n`timescale 1ns / 1ps\n`ifdef (A && !B)\nx > 0;\n// low\n`elsif C\n"
                                     "x < 9;\n`else\n`endif\nx >\n`endif")
assert PPText.ToText is SVText.ToText and validate(comment(" c"), (), SVC.DIALECT) == []

<a id="SVD-06"></a>
## SVD-06 · Invalid syntax: error and missing among its terms, validated and printed as written, an error with no text refused by name

In [ ]:
error = lambda u: u.error(lambda e: e.text("`define"))  # noqa: E731
missing = lambda u: u.missing(lambda m: m.kind("macrousage"))  # noqa: E731
assert validate(error) == ["invalid syntax '`define'"] and validate(missing) == ["missing macrousage"]
assert text(error) == "`define" and text(missing) == ""
with raises(ValueError, match="SystemVerilog Preprocessor error has no text to print"):
    text(lambda u: u.error(lambda e: e))

<a id="SVD-07"></a>
## SVD-07 · Its editions: Verilog's and SystemVerilog's eight, each by its designation and common name, each with every one of its kinds

In [ ]:
own = store.Instances.Expressions.SystemVerilog.Preprocessor
assert list(FS.standards(own)) == [designation for designation, *_ in SV.EDITIONS]
assert [FS.tags(s) for s in FS.standards(own).values()] == [(designation, common) for designation, _, _, common, _ in SV.EDITIONS]
assert FS.properties(FS.find(own, "Verilog-1995")) == {"bodies": ("org.ieee",), "number": "1364", "revision": ("year", 1995),
                                                       "names": ("Verilog-1995",)}
assert FS.find(own, "IEEE Std 1800-2023") is FS.find(own, "SystemVerilog-2023") and PP.SINCE == {}
every = [kinds[tag] for tag in PP.TAGS]  # each of its kinds in every edition; helpers and invalid syntax in none
assert all(FS.kinds(s) == every for s in FS.standards(own).values())
assert list(FS.of(own, kinds["comment"])) == list(FS.standards(own)) and FS.of(own, registered("ElsifDirective")) == {}
store.transaction(SVPreprocessor.define)  # run again, it writes nothing
assert len(FS.standards(own)) == 8

<a id="SVD-08"></a>
## SVD-08 · Printing: each kind as mbse-programs' printer writes it, a directive at the start of its line, from any holder of the family

In [ ]:
for spec, source in [
    (lambda u: u.macrousage(lambda m: m.name(spelled("WIDTH"))), "`WIDTH"),
    (lambda u: u.macrousage(lambda m: m.name(spelled("MAX")).arguments("a, b")), "`MAX(a, b)"),
    (lambda u: u.includedirective(lambda d: d.path("defs.svh")), '`include "defs.svh"'),
    (lambda u: u.includedirective(lambda d: d.path("uvm.svh").system(True)), "`include <uvm.svh>"),
    (lambda u: u.definedirective(lambda d: d.name(spelled("W"))), "`define W"),
    (lambda u: u.definedirective(lambda d: d.name(spelled("W")).body("8")), "`define W 8"),
    (lambda u: u.definedirective(lambda d: d.name(spelled("MAX")).function_like(True).parameters([spelled("a"), spelled("b")])
        .body("((a) > (b) ? (a) : (b))")), "`define MAX(a, b) ((a) > (b) ? (a) : (b))"),
    (lambda u: u.definedirective(lambda d: d.name(spelled("F")).function_like(True).body("1 \\\n+ 2")), "`define F() 1 \\\n+ 2"),
    (lambda u: u.undefdirective(lambda d: d.name(spelled("W"))), "`undef W"),
    (lambda u: u.timescaledirective(lambda d: d.unit("1ns").precision("10ps")), "`timescale 1ns / 10ps"),
    (lambda u: u.defaultnettypedirective(lambda d: d.net_type("none")), "`default_nettype none"),
    (lambda u: u.ifdefdirective(lambda d: d.name(spelled("A"))), "`ifdef A\n`endif"),
    (lambda u: u.ifdefdirective(lambda d: d.negated(True).name(spelled("A")).items([lambda i: i.disabledtext(lambda t: t.text("x < 1;\nx > 2;"))])
        .alternative([SV.binary("==", x, 1)]).has_else(True)), "`ifndef A\nx < 1;\nx > 2;\n`else\nx == 1;\n`endif"),
    (ifdef, "`ifdef (A && !B)\nx > 0;\n// low\n`elsif C\nx < 9;\n`else\n`endif"),
    (lambda u: u.disabledtext(lambda t: t.text("a;\nb;")), "a;\nb;"),
    (lambda u: u.otherdirective(lambda d: d.text("`celldefine")), "`celldefine"),
    (comment(" note"), "// note"), (comment(" note\n * more ", True), "/* note\n * more */"),
    (lambda u: u.comment(lambda c: c.trailing(True)), "//"),
]:
    assert text(spec) == source, (source, text(spec))
assert text(lambda u: u.macrousage(lambda m: m.name(spelled("W"))), SV.DIALECT) == "`W"  # from Expressions' holder
assert text(SV.binary("+", lambda u: u.macrousage(lambda m: m.name(spelled("W"))), 1), SV.DIALECT) == "`W + 1"
for spec in [lambda u: u.ifdefdirective(lambda d: d), lambda u: u.ifdefdirective(lambda d: d.name(spelled("A")).branches([lambda b: b]))]:
    with raises(ValueError, match="has no name or condition to print"):
        text(spec)

<a id="SVD-09"></a>
## SVD-09 · Evaluation: none (Q11); each of its kinds is refused by name, wherever it stands

In [ ]:
for tag, spec in [
    ("macrousage", lambda u: u.macrousage(lambda m: m.name(spelled("W")))),
    ("includedirective", lambda u: u.includedirective(lambda d: d.path("a.svh"))),
    ("definedirective", lambda u: u.definedirective(lambda d: d.name(spelled("W")))),
    ("undefdirective", lambda u: u.undefdirective(lambda d: d.name(spelled("W")))),
    ("timescaledirective", lambda u: u.timescaledirective(lambda d: d.unit("1ns").precision("1ps"))),
    ("defaultnettypedirective", lambda u: u.defaultnettypedirective(lambda d: d.net_type("wire"))),
    ("ifdefdirective", ifdef), ("disabledtext", lambda u: u.disabledtext(lambda t: t.text("a"))),
    ("otherdirective", lambda u: u.otherdirective(lambda d: d.text("`resetall"))), ("comment", comment(" c")),
]:
    with raises(NotImplementedError, match=f"SystemVerilog Preprocessor {tag} is not evaluated"):
        SVV.OfAny(term(spec), {"x": 1})
with raises(NotImplementedError, match="SystemVerilog Preprocessor macrousage is not evaluated"):  # within an expression
    SVV.OfAny(term(SV.binary("+", lambda u: u.macrousage(lambda m: m.name(spelled("W"))), 1), SV.DIALECT))